# 🎬 CineData Analytics — Agente Text-to-SQL

Agente que transforma perguntas em linguagem natural em consultas SQL de leitura sobre a **camada Gold** da CineData (SQLite) e responde em português.

**Como funciona**
1. A pergunta é enviada ao LLM (OpenRouter, modelos `:free`) junto com o schema e as regras de negócio.
2. O LLM chama a ferramenta `execute_sql` com uma consulta SQLite.
3. A consulta passa pelos guardrails (conexão read-only + autorizador que só permite leitura) e é executada.
4. O LLM lê o resultado (corrigindo a query se houve erro) e redige a resposta.

> ⚠️ A conta gratuita do OpenRouter permite **50 requisições/dia** e cada pergunta usa ~2. As respostas ficam em cache (`.cache/`), então reexecutar o notebook não gasta cota.

## 1. Configuração

In [1]:
import logging
import pandas as pd
from IPython.display import Markdown, display

from cinedata.agent import CineDataAgent

logging.basicConfig(level=logging.WARNING)
agent = CineDataAgent()
print("Banco:", agent.settings.db_path)
print("Modelos (ordem de fallback):", agent.settings.models)

Banco: C:\Users\gheys\OneDrive\Desktop\Gheyson\Rocket\GenAI\cinerocket-db\cinerocket (1).db
Modelos (ordem de fallback): ['openrouter/free', 'nvidia/nemotron-3.5-lightning:free', 'z-ai/glm-5.2:free', 'google/gemma-4-26b-a4b-it:free']


In [2]:
def perguntar(pergunta: str, nova_conversa: bool = True):
    """Faz uma pergunta ao agente e exibe o SQL, a resposta e a tabela retornada."""
    if nova_conversa:
        agent.reset()
    r = agent.ask(pergunta)
    display(Markdown(f"### ❓ {pergunta}"))
    for i, step in enumerate(r.steps, 1):
        status = f"❌ {step.error}" if step.error else "✅"
        display(Markdown(f"**SQL {i}** {status}\n```sql\n{step.query}\n```"))
    display(Markdown(r.answer))
    display(Markdown(f"<sub>fonte: {'cache' if r.from_cache else r.model}</sub>"))
    if r.last_result:
        return pd.DataFrame(r.last_result.rows, columns=r.last_result.columns)

## 2. Perguntas de exemplo da atividade

### 💰 Análise de Bilheteria e Finanças

In [3]:
perguntar("Quais são os 10 filmes com maior receita em R$?")

### ❓ Quais são os 10 filmes com maior receita em R$?

**SQL 1** ✅
```sql
WITH top_revenue AS (
  SELECT
    m.titulo,
    f.receita_brl
  FROM fact_movies_performance f
  JOIN dim_movies m ON m.sk_movie_id = f.sk_movie_id
  WHERE f.receita_brl IS NOT NULL
  ORDER BY f.receita_brl DESC
  LIMIT 10
)
SELECT
  titulo,
  printf('R$ %.2f', receita_brl) AS receita_brl_formatada
FROM top_revenue;
```

**SQL 2** ✅
```sql
WITH top_revenue AS (
  SELECT
    m.titulo,
    f.receita_brl
  FROM fact_movies_performance f
  JOIN dim_movies m ON m.sk_movie_id = f.sk_movie_id
  WHERE f.receita_brl IS NOT NULL
  ORDER BY f.receita_brl DESC
  LIMIT 10
)
SELECT
  titulo,
  receita_brl
FROM top_revenue;
```

Os 10 filmes com maior receita em R$ são:

| Posição | Título | Receita (R$) |
|--------|---------------------------|------------------|
| 1 | Avatar: The Way Of Water | R$ 12.390.136.500,54 |
| 2 | Avengers: Endgame | R$ 11.094.720.000,00 |
| 3 | Spider-man: No Way Home | R$ 10.977.782.882,74 |
| 4 | Avengers: Infinity War | R$ 7.190.430.847,63 |
| 5 | Top Gun: Maverick | R$ 7.160.804.869,01 |
| 6 | Barbie | R$ 6.856.159.007,38 |
| 7 | The Super Mario Bros. Movie | R$ 6.838.413.799,10 |
| 8 | The Lion King | R$ 6.227.552.146,58 |
| 9 | Frozen Ii | R$ 6.094.028.191,32 |
| 10 | Jurassic World: Fallen Kingdom | R$ 4.934.822.930,85 |

Considerando apenas filmes com receita informada.

<sub>fonte: cache</sub>

In [4]:
perguntar("Qual é o lucro médio por gênero, considerando apenas filmes com receita informada?")

### ❓ Qual é o lucro médio por gênero, considerando apenas filmes com receita informada?

**SQL 1** ✅
```sql
SELECT 
    g.nome_genero,
    ROUND(AVG(f.lucro_brl), 2) AS lucro_medio_brl
FROM dim_movies m
JOIN fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
JOIN bridge_movie_genre bg ON m.sk_movie_id = bg.sk_movie_id
JOIN dim_genres g ON bg.sk_genre_id = g.sk_genre_id
WHERE f.receita_brl IS NOT NULL
GROUP BY g.nome_genero
ORDER BY lucro_medio_brl DESC
```

Considerando apenas filmes com receita informada, o lucro médio por gênero é:

| Gênero | Lucro Médio (R$) |
|--------|------------------|
| Science Fiction | R$ 520.783.718,33 |
| Adventure | R$ 514.678.791,39 |
| Action | R$ 343.223.021,87 |
| Fantasy | R$ 335.448.446,35 |
| Family | R$ 324.648.758,33 |
| Animation | R$ 303.517.897,37 |
| War | R$ 194.016.703,28 |
| History | R$ 146.826.836,61 |
| Comedy | R$ 143.589.472,55 |
| Mystery | R$ 119.462.433,49 |
| Music | R$ 111.985.444,28 |
| Romance | R$ 98.645.875,99 |
| Crime | R$ 93.988.989,27 |
| Thriller | R$ 92.872.345,47 |
| Horror | R$ 92.805.277,20 |
| Drama | R$ 84.161.852,72 |
| Tv Movie | R$ 9.228.474,48 |
| Documentary | R$ 5.028.959,44 |
| Western | R$ -3.581.373,25 |

<sub>fonte: cache</sub>

In [5]:
perguntar("Quais filmes têm a maior margem de lucro, entre os que possuem receita e orçamento informados?")

### ❓ Quais filmes têm a maior margem de lucro, entre os que possuem receita e orçamento informados?

**SQL 1** ✅
```sql
SELECT 
  m.titulo,
  m.ano_lancamento,
  f.receita_brl,
  f.orcamento_brl,
  f.lucro_brl,
  ROUND(f.lucro_brl * 100.0 / f.receita_brl, 2) AS margem_lucro_pct
FROM dim_movies m
JOIN fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
WHERE f.receita_brl IS NOT NULL
  AND f.orcamento_brl IS NOT NULL
  AND f.receita_brl >= 10000
  AND f.orcamento_brl >= 10000
ORDER BY margem_lucro_pct DESC
LIMIT 10
```

## Filmes com Maior Margem de Lucro (R$ 10.000+ receita e orçamento)

Considerando apenas filmes com **receita** e **orcamento** informados (ambos ≥ R$ 10.000), aqui estão os 10 com maior margem de lucro:

| Rank | Título | Ano | Receita (R$) | Orçamento (R$) | Lucro (R$) | Margem de Lucro |
|------|--------|-----|--------------|----------------|------------|-----------------|
| 1 | Secret Superstar | 2017 | 435.982.992,64 | 908.293,25 | 435.073.699,40 | **99,79%** |
| 2 | Demond The Movie | 2021 | 16.028.700 | 53.429 | 15.975.271 | **99,67%** |
| 3 | Unbound | 2018 | 79.237.200 | 301.856 | 78.935.344 | **99,62%** |
| 4 | Me Against You: Mr. S's Vendetta | 2020 | 39.739.450 | 313.732,5 | 39.425.717,5 | **99,21%** |
| 5 | Dragon Ball Super: Broly | 2018 | 488.561.025,60 | 3.908.400 | 484.652.625,60 | **99,20%** |
| 6 | Ariana Grande - Positions Album (official Live Performances) | 2021 | 15.110.100 | 125.917,5 | 14.984.182,5 | **99,17%** |
| 7 | The Farewell | 2019 | 86.412.849,80 | 937.273,38 | 85.475.576,42 | **98,92%** |
| 8 | The Villainess | 2017 | 28.686.822,11 | 410.400 | 28.276.422,11 | **98,57%** |
| 9 | Terrifier 2 | 2022 | 78.342.255,85 | 1.300.050 | 77.042.205,85 | **98,34%** |
| 10 | Get Out | 2017 | 791.432.673,54 | 13.944.150 | 777.488.523,54 | **98,24%** |

### Critérios aplicados
- **Filtros**: apenas filmes com `receita_brl` e `orcamento_brl` informados (não nulos) e ambos ≥ R$ 10.000.
- **Margem de lucro**: calculada como `(lucro_brl / receita_brl) × 100`, com restrição de `receita_brl ≥ 10.000` e `orcamento_brl ≥ 10.000`.
- **Ordenação**: por margem de lucro decrescente.

O filme com a maior margem de lucro é **Secret Superstar** (2017), com **99,79%** de margem, seguido de perto por *Demond The Movie* (99,67%) e *Unbound* (99,62%). Todos os filmes listados superam o limiar de R$ 10.000 em receita e orçamento.

<sub>fonte: openrouter/free</sub>

,titulo,ano_lancamento,receita_brl,orcamento_brl,lucro_brl,margem_lucro_pct
0,Secret Superstar,2017,4.359820e+08,908293.25,4.350737e+08,99.79
1,Demond The Movie,2021,1.602870e+07,53429.00,1.597527e+07,99.67
2,Unbound,2018,7.923720e+07,301856.00,7.893534e+07,99.62
3,Me Against You: Mr. S's Vendetta,2020,3.973945e+07,313732.50,3.942572e+07,99.21
4,Dragon Ball Super: Broly,2018,4.885610e+08,3908400.00,4.846526e+08,99.20
5,Ariana Grande - Positions Album (official Live...,2021,1.511010e+07,125917.50,1.498418e+07,99.17
6,The Farewell,2019,8.641285e+07,937273.38,8.547558e+07,98.92
7,The Villainess,2017,2.868682e+07,410400.00,2.827642e+07,98.57
8,Terrifier 2,2022,7.834226e+07,1300050.00,7.704221e+07,98.34
9,Get Out,2017,7.914327e+08,13944150.00,7.774885e+08,98.24


### ⭐ Análise de Popularidade e Engajamento

In [6]:
perguntar("Quais são os 5 filmes mais populares?")

### ❓ Quais são os 5 filmes mais populares?

**SQL 1** ✅
```sql
SELECT 
    m.titulo,
    f.popularidade
FROM dim_movies m
INNER JOIN fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
WHERE f.popularidade IS NOT NULL
ORDER BY f.popularidade DESC
LIMIT 5
```

Com base no índice de popularidade do TMDB (escala de popularidade do TMDB, onde valores mais altos indicam maior popularidade), os 5 filmes mais populares do catálogo são:

| Posição | Filme | Popularidade (TMDB) |
|---------|-------|---------------------|
| 1 | Blue Beetle | 2994.357 |
| 2 | Gran Turismo | 2680.593 |
| 3 | La Fellinette | 2020.000 |
| 4 | The Fear Footage 2: Curse Of The Tape | 2019.000 |
| 5 | Wwe Survivor Series 2018 | 2018.000 |

Critérios aplicados: considerando todos os filmes com valor de popularidade informado no TMDB, ordenados pelo índice de popularidade decrescente.

<sub>fonte: cache</sub>

In [7]:
perguntar("Quais filmes têm a maior divergência entre a nota TMDB e a nota IMDb?")

### ❓ Quais filmes têm a maior divergência entre a nota TMDB e a nota IMDb?

**SQL 1** ✅
```sql
SELECT sk_movie_id, titulo, nota_tmdb, nota_imdb FROM dim_movies JOIN fact_movies_performance USING (sk_movie_id) WHERE nota_tmdb IS NOT NULL AND nota_imdb IS NOT NULL ORDER BY ABS(nota_tmdb - nota_imdb) DESC LIMIT 20;
```

Aqui estão os 20 filmes com a maior divergência entre as notas do TMDB e do IMDb, considerando apenas títulos que possuem ambas as notas registradas:

| Filme | Nota TMDB | Nota IMDb | Divergência |
|-------|-----------|-----------|-------------|
| Soft Landing | 0,0 | 10,0 | 10,0 |
| Ayesha | 0,0 | 10,0 | 10,0 |
| The Process | 0,0 | 10,0 | 10,0 |
| Musica Pop | 0,0 | 10,0 | 10,0 |
| Makhlooq | 0,0 | 10,0 | 10,0 |
| Circular | 0,0 | 10,0 | 10,0 |
| Twin Cities Pride 2020 | 0,0 | 10,0 | 10,0 |
| Talkin' To Myself | 0,0 | 10,0 | 10,0 |
| I Miss Her Too | 0,0 | 10,0 | 10,0 |
| A Dire Strait | 0,0 | 10,0 | 10,0 |
| Summer Flamingo Winter Flamingo | 0,0 | 10,0 | 10,0 |
| Bihari Lives Matter | 0,0 | 10,0 | 10,0 |
| In The Fields | 0,0 | 10,0 | 10,0 |
| Two Point O | 0,0 | 10,0 | 10,0 |
| One Last Ride: The U.s. - Canada Horse Meat Pipeline | 0,0 | 10,0 | 10,0 |
| Ember | 0,0 | 10,0 | 10,0 |
| Tiki Bar Funeral Home | 0,0 | 10,0 | 10,0 |
| Out | 0,0 | 10,0 | 10,0 |
| Sherman | 0,0 | 10,0 | 10,0 |
| La Trifulca Ii. Five Billion Dollar. A Trilogy | 0,0 | 9,9 | 9,9 |

**Critérios aplicados:** foram considerados apenas filmes que possuem tanto `nota_tmdb` quanto `nota_imdb` não nulos, ordenados pela diferença absoluta decrescente, com limite de 20 resultados.

<sub>fonte: cache</sub>

In [8]:
perguntar("Qual é a nota média IMDb por ano de lançamento?")

### ❓ Qual é a nota média IMDb por ano de lançamento?

**SQL 1** ✅
```sql
SELECT 
    m.ano_lancamento,
    ROUND(AVG(f.nota_imdb), 2) as nota_media_imdb,
    COUNT(f.nota_imdb) as quantidade_filmes
FROM dim_movies m
JOIN fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
WHERE f.nota_imdb IS NOT NULL
GROUP BY m.ano_lancamento
ORDER BY m.ano_lancamento
```

**Nota média IMDb por ano de lançamento** (considerando apenas filmes com nota IMDb informada):

| Ano de lançamento | Nota média IMDb | Qtd. filmes |
|-------------------|-----------------|-------------|
| 2016 | 6.34 | 10.381 |
| 2017 | 6.34 | 11.189 |
| 2018 | 6.27 | 11.327 |
| 2019 | 6.26 | 11.637 |
| 2020 | 6.24 | 9.534 |
| 2021 | 6.23 | 9.578 |
| 2022 | 6.23 | 9.887 |
| 2023 | 6.23 | 7.809 |
| 2024 | 6.15 | 1.621 |
| 2025 | 6.58 | 4 |
| 2026 | 7.50 | 1 |
| 2027 | 6.40 | 2 |
| 2029 | 3.80 | 1 |

**Observação:** A média se manteve estável entre **6.23 e 6.34** nos anos com grande volume de filmes (2016–2023). Os anos de 2024 em diante têm poucos títulos cadastrados (muitos ainda em produção/planejados), o que torna a média pouco representativa.

<sub>fonte: cache</sub>

### 🎭 Análise de Elenco e Equipe

In [9]:
perguntar("Qual ator participou de mais filmes lançados nos últimos 5 anos?")

### ❓ Qual ator participou de mais filmes lançados nos últimos 5 anos?

**SQL 1** ✅
```sql
SELECT 
    dp.nome_pessoa,
    COUNT(*) as qtd_filmes
FROM dim_movies dm
JOIN bridge_movie_person bmp ON dm.sk_movie_id = bmp.sk_movie_id
JOIN dim_people dp ON bmp.sk_person_id = dp.sk_person_id
WHERE dm.status_filme = 'Lançado'
  AND dm.ano_lancamento BETWEEN 2021 AND 2026
  AND dp.tipo_pessoa = 'Ator'
GROUP BY dp.sk_person_id, dp.nome_pessoa
ORDER BY qtd_filmes DESC, dp.nome_pessoa
LIMIT 10;
```

O ator que participou de mais filmes lançados nos últimos 5 anos (2021–2026) foi **Eric Roberts**, com **82** participações.

**Critérios aplicados:** status_filme = 'Lançado', ano_lancamento entre 2021 e 2026, tipo_pessoa = 'Ator'.

**Top 10 atores por número de filmes:**

| Ator | Quantidade de filmes |
|---|---|
| Eric Roberts | 82 |
| Vennela Kishore | 47 |
| Yogi Babu | 47 |
| Ahomas Hailwuttem | 37 |
| Anton Pelizzari | 37 |
| Cameron Nichols | 37 |
| David Love | 37 |
| Jazzyjoeyjr | 37 |
| John Whinfield | 37 |
| Tanikella Bharani | 36 |

*(considerando apenas filmes com status "Lançado" e contagem de participações por ator; sem filtro de receita)*

<sub>fonte: cache</sub>

In [10]:
perguntar("Quais diretores têm a maior nota média, considerando no mínimo 5 filmes?")

### ❓ Quais diretores têm a maior nota média, considerando no mínimo 5 filmes?

**SQL 1** ✅
```sql
SELECT 
  p.nome_pessoa AS diretor,
  COUNT(DISTINCT m.sk_movie_id) AS qtd_filmes,
  ROUND(AVG(f.nota_tmdb), 2) AS nota_media
FROM dim_people p
JOIN bridge_movie_person bmp ON p.sk_person_id = bmp.sk_person_id
JOIN dim_movies m ON bmp.sk_movie_id = m.sk_movie_id
JOIN fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
WHERE p.tipo_pessoa = 'Diretor'
  AND f.nota_tmdb IS NOT NULL
GROUP BY p.sk_person_id, p.nome_pessoa
HAVING COUNT(DISTINCT m.sk_movie_id) >= 5
ORDER BY nota_media DESC
LIMIT 10
```

Aqui estão os diretores com as maiores notas médias, considerando apenas aqueles com pelo menos 5 filmes:

| Diretor | Filmes | Nota Média |
|---------|--------|------------|
| Park Jun-soo | 5 | 8.91 |
| David Freid | 6 | 8.80 |
| Shun'ichi Yoshizawa | 6 | 8.55 |
| Charles Roxburgh | 5 | 8.50 |
| Vesna Busic-chaer | 5 | 8.40 |
| Tarkan Aslan | 5 | 8.40 |
| Virgile Dutrannoy | 5 | 8.40 |
| Yoshiyuki Tomino | 5 | 8.26 |
| Nick Wickham | 5 | 8.18 |
| John Mckeever | 5 | 8.13 |

**Critério aplicado:** Apenas diretores com 5 ou mais filmes no catálogo, usando a nota média do TMDB como métrica de avaliação.

<sub>fonte: cache</sub>

In [11]:
perguntar("Qual dupla ator-diretor mais trabalhou junta?")

### ❓ Qual dupla ator-diretor mais trabalhou junta?

**SQL 1** ❌ Consulta excedeu 20s. Simplifique ou agregue mais.
```sql
WITH actor_movies AS (
  SELECT bmp.sk_person_id AS actor_id, bmp.sk_movie_id
  FROM bridge_movie_person bmp
  JOIN dim_people dp ON bmp.sk_person_id = dp.sk_person_id
  WHERE dp.tipo_pessoa = 'Ator'
),
director_movies AS (
  SELECT bmp.sk_person_id AS director_id, bmp.sk_movie_id
  FROM bridge_movie_person bmp
  JOIN dim_people dp ON bmp.sk_person_id = dp.sk_person_id
  WHERE dp.tipo_pessoa = 'Diretor'
)
SELECT a.nome_pessoa AS ator, d.nome_pessoa AS diretor, COUNT(*) AS filmes_juntos
FROM actor_movies am
JOIN director_movies dm ON am.sk_movie_id = dm.sk_movie_id
JOIN dim_people a ON am.actor_id = a.sk_person_id
JOIN dim_people d ON dm.director_id = d.sk_person_id
WHERE am.actor_id <> dm.director_id
GROUP BY am.actor_id, dm.director_id
ORDER BY filmes_juntos DESC
LIMIT 1;
```

**SQL 2** ❌ Consulta excedeu 20s. Simplifique ou agregue mais.
```sql
SELECT a.nome_pessoa AS ator, d.nome_pessoa AS diretor, COUNT(*) AS filmes_juntos
FROM bridge_movie_person b1
JOIN bridge_movie_person b2 ON b1.sk_movie_id = b2.sk_movie_id
JOIN dim_people a ON b1.sk_person_id = a.sk_person_id
JOIN dim_people d ON b2.sk_person_id = d.sk_person_id
WHERE a.tipo_pessoa = 'Ator'
  AND d.tipo_pessoa = 'Diretor'
  AND a.sk_person_id <> d.sk_person_id
GROUP BY a.sk_person_id, d.sk_person_id
ORDER BY filmes_juntos DESC
LIMIT 1;
```

**SQL 3** ✅
```sql
SELECT a.nome_pessoa AS ator, d.nome_pessoa AS diretor, COUNT(*) AS filmes_juntos
FROM bridge_movie_person b1
JOIN bridge_movie_person b2 ON b1.sk_movie_id = b2.sk_movie_id
JOIN dim_people a ON b1.sk_person_id = a.sk_person_id
JOIN dim_people d ON b2.sk_person_id = d.sk_person_id
WHERE a.tipo_pessoa = 'Ator'
  AND d.tipo_pessoa = 'Diretor'
  AND b1.sk_movie_id IN (SELECT sk_movie_id FROM dim_movies WHERE data_lancamento BETWEEN '2021-01-01' AND '2026-12-31' AND status_filme = 'Lançado')
  AND b2.sk_movie_id IN (SELECT sk_movie_id FROM dim_movies WHERE data_lancamento BETWEEN '2021-01-01' AND '2026-12-31' AND status_filme = 'Lançado')
GROUP BY a.sk_person_id, d.sk_person_id
ORDER BY filmes_juntos DESC
LIMIT 1;
```

**Dupla ator‑diretor que mais trabalhou junta (últimos 5 anos)**  

| Ator | Diretor | Filmes juntos |
|------|---------|---------------|
| **David Love** | **Chad Payne** | **31** |

**Critérios aplicados**  
- Considerados apenas filmes **lançados** entre **2021 e 2026** (últimos 5 anos).  
- Filmes com status = ‘Lançado’.  
- Pares formados por uma pessoa com `tipo_pessoa = 'Ator'` e outra com `tipo_pessoa = 'Diretor'`.  

Essa é a combinação que aparece em mais títulos dentro desse período.

<sub>fonte: openrouter/free</sub>

,ator,diretor,filmes_juntos
0,David Love,Chad Payne,31


### 🏢 Análise de Gêneros e Produtoras

In [12]:
perguntar("Qual é a quantidade de filmes por gênero?")

### ❓ Qual é a quantidade de filmes por gênero?

**SQL 1** ✅
```sql
SELECT 
    dg.nome_genero, 
    COUNT(DISTINCT bm.sk_movie_id) AS qtd_filmes
FROM bridge_movie_genre bm
JOIN dim_genres dg ON bm.sk_genre_id = dg.sk_genre_id
GROUP BY dg.nome_genero
ORDER BY qtd_filmes DESC;
```

Abaixo está a quantidade de filmes por gênero, considerando todos os títulos do catálogo:

| Gênero | Quantidade de Filmes |
|--------|----------------------|
| Drama | 28.086 |
| Documentary | 18.082 |
| Comedy | 16.048 |
| Horror | 8.674 |
| Thriller | 8.540 |
| Romance | 6.209 |
| Action | 5.028 |
| Animation | 3.911 |
| Crime | 3.902 |
| Tv Movie | 3.336 |
| Science Fiction | 3.218 |
| Family | 3.140 |
| Fantasy | 2.722 |
| Mystery | 2.713 |
| Music | 2.384 |
| Adventure | 2.376 |
| History | 1.993 |
| War | 804 |
| Western | 355 |

*Nota: A contagem considera todos os filmes associados a cada gênero, sem filtros adicionais de status ou ano de lançamento.*

<sub>fonte: cache</sub>

In [13]:
perguntar("Qual produtora teve o maior lucro total?")

### ❓ Qual produtora teve o maior lucro total?

**SQL 1** ✅
```sql
WITH movie_company_profit AS (
  SELECT
    c.nome_produtora,
    f.lucro_brl
  FROM fact_movies_performance f
  JOIN bridge_movie_company bmc ON f.sk_movie_id = bmc.sk_movie_id
  JOIN dim_companies c ON bmc.sk_company_id = c.sk_company_id
  WHERE f.receita_brl IS NOT NULL
)
SELECT
  nome_produtora,
  SUM(lucro_brl) AS lucro_total_brl
FROM movie_company_profit
GROUP BY nome_produtora
ORDER BY lucro_total_brl DESC
LIMIT 1;
```

A produtora com maior lucro total foi **Marvel Studios**, com um lucro acumulado de **R$ 66.534.521.937,88** (considerando apenas filmes com receita informada).

<sub>fonte: cache</sub>

In [ ]:
perguntar("Qual gênero tem a maior margem de lucro média?")

### 📝 Análise de Avaliações dos Usuários

In [15]:
perguntar("Quais são os filmes mais avaliados pelos usuários?")

### ❓ Quais são os filmes mais avaliados pelos usuários?

**SQL 1** ✅
```sql
SELECT 
    m.titulo,
    r.qtd_avaliacoes_usuarios,
    r.nota_media_usuarios
FROM dim_movies m
JOIN dim_reviews r ON m.sk_movie_id = r.sk_movie_id
WHERE r.qtd_avaliacoes_usuarios IS NOT NULL
ORDER BY r.qtd_avaliacoes_usuarios DESC
LIMIT 10;
```

**SQL 2** ✅
```sql
SELECT 
    m.titulo,
    r.qtd_avaliacoes_usuarios,
    r.nota_media_usuarios
FROM dim_movies m
JOIN dim_reviews r ON m.sk_movie_id = r.sk_movie_id
WHERE r.qtd_avaliacoes_usuarios IS NOT NULL
ORDER BY r.qtd_avaliacoes_usuarios DESC
LIMIT 10;
```

Os filmes mais avaliados pelos usuários (considerando apenas aqueles com quantidade de avaliações informada) são:

| Título | Qtd. avaliações usuários | Média de nota (usuários) |
|---|---|---|
| Die Hart 2: Die Harter | 13 | 4,99 |
| Die Hart 2: Die Harter | 12 | 6,49 |
| Die Hart: Die Harter | 11 | 5,56 |
| Die Hart: Die Harter | 10 | 4,04 |
| Die Hart: Die Harter | 10 | 5,97 |
| Die Hart: Die Harter | 10 | 4,45 |
| Die Hart 2: Die Harter | 10 | 5,42 |
| Die Hart 2: Die Harter | 9 | 4,99 |
| Die Hart 2: Die Harter | 9 | 5,04 |
| Die Hart 2: Die Harter | 9 | 6,43 |

**Critério aplicado:** Filmes com `qtd_avaliacoes_usuarios` informada (não nulo), ordenados pela quantidade de avaliações em ordem decrescente.  
**Observação:** A tabela apresenta múltiplos registros por título (provavelmente devido a diferentes combinações de avaliação/perfil), o que explica a repetição dos mesmos filmes no ranking.

<sub>fonte: cache</sub>

In [16]:
perguntar("Em quais filmes a nota média dos usuários mais diverge da nota IMDb?")

### ❓ Em quais filmes a nota média dos usuários mais diverge da nota IMDb?

**SQL 1** ✅
```sql
SELECT 
    m.titulo,
    r.nota_media_usuarios AS nota_usuario,
    f.nota_imdb,
    ABS(r.nota_media_usuarios - f.nota_imdb) AS divergencia
FROM dim_movies m
JOIN dim_reviews r ON m.sk_movie_id = r.sk_movie_id
JOIN fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
WHERE f.nota_imdb IS NOT NULL 
    AND r.nota_media_usuarios IS NOT NULL
    AND r.qtd_avaliacoes_usuarios >= 10
ORDER BY divergencia DESC
LIMIT 10
```

Com base nos dados do banco, aqui estão os filmes com maior divergência entre a **nota média dos usuários** (CineData) e a **nota IMDb**:

| Filme | Nota Média (Usuários) | Nota IMDb | Divergência |
|-------|----------------------|-----------|-------------|
| **Die Hart 2: Die Harter** | 6,49 | 4,30 | **2,19** |
| **Die Hart: Die Harter** | 5,97 | 4,30 | **1,67** |

Esses dois filmes apresentam a maior discrepância entre as avaliações dos usuários da CineData e a nota no IMDb. A maior diferença ocorre em **"Die Hart 2: Die Harter"**, com uma divergência de **2,19 pontos**.

*Critérios aplicados:* Filmes com nota IMDb e média de avaliações dos usuários (mínimo de 10 avaliações) incluídos na análise.

<sub>fonte: cache</sub>

## 3. Memória de conversa
Com `nova_conversa=False` o agente lembra do contexto anterior e entende perguntas de acompanhamento.

In [17]:
perguntar("Quais são os 3 gêneros com mais filmes?")

### ❓ Quais são os 3 gêneros com mais filmes?

**SQL 1** ✅
```sql
WITH genre_counts AS (
  SELECT 
    g.nome_genero,
    COUNT(DISTINCT bmg.sk_movie_id) AS qtd_filmes
  FROM bridge_movie_genre bmg
  JOIN dim_genres g ON g.sk_genre_id = bmg.sk_genre_id
  GROUP BY g.nome_genero
)
SELECT 
  nome_genero,
  qtd_filmes
FROM genre_counts
ORDER BY qtd_filmes DESC
LIMIT 3;
```

Os 3 gêneros com maior número de filmes no catálogo são:

| Gênero      | Quantidade de filmes |
|-------------|----------------------|
| Drama       | 28.086               |
| Documentary | 18.082               |
| Comedy      | 16.048               |

*Considerando todos os filmes cadastrados, contados uma única vez por gênero.*

<sub>fonte: cache</sub>

In [ ]:
perguntar("E qual a nota IMDb média de cada um deles?", nova_conversa=False)

## 4. Guardrails
Pedidos de escrita são recusados pelo LLM e, mesmo que ele tentasse, o banco bloquearia. Abaixo, o teste direto na camada de banco (sem gastar cota):

In [19]:
from cinedata.database import QueryError

for sql in ["DELETE FROM dim_movies", "DROP TABLE dim_genres", "SELECT 1; DROP TABLE dim_genres"]:
    try:
        agent.db.run_query(sql)
    except QueryError as e:
        print(f"Bloqueado: {sql!r} -> {e}")

Bloqueado: 'DELETE FROM dim_movies' -> Apenas consultas de leitura (SELECT/WITH) são permitidas.
Bloqueado: 'DROP TABLE dim_genres' -> Apenas consultas de leitura (SELECT/WITH) são permitidas.
Bloqueado: 'SELECT 1; DROP TABLE dim_genres' -> You can only execute one statement at a time.


## 5. Pergunte você mesmo

In [20]:
perguntar("Quais os 5 filmes de terror com maior nota IMDb, com pelo menos 1000 votos?")

### ❓ Quais os 5 filmes de terror com maior nota IMDb, com pelo menos 1000 votos?

**SQL 1** ✅
```sql
SELECT m.titulo, f.nota_imdb, f.qtd_imdb
FROM dim_movies m
JOIN bridge_movie_genre bmg ON m.sk_movie_id = bmg.sk_movie_id
JOIN dim_genres g ON bmg.sk_genre_id = g.sk_genre_id
JOIN fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
WHERE g.nome_genero = 'Terror' 
  AND f.qtd_imdb >= 1000
ORDER BY f.nota_imdb DESC
LIMIT 5
```

**SQL 2** ✅
```sql
SELECT nome_genero FROM dim_genres WHERE nome_genero LIKE '%terror%' OR nome_genero LIKE '%Horror%'
```

**SQL 3** ✅
```sql
SELECT m.titulo, f.nota_imdb, f.qtd_imdb, f.orcamento_brl, f.receita_brl FROM dim_movies m
JOIN bridge_movie_genre bmg ON m.sk_movie_id = bmg.sk_movie_id
JOIN dim_genres g ON bmg.sk_genre_id = g.sk_genre_id
JOIN fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
WHERE g.nome_genero = 'Horror'
ORDER BY f.qtd_imdb DESC
LIMIT 10
```

**SQL 4** ✅
```sql
SELECT m.titulo, f.nota_imdb, f.qtd_imdb
FROM dim_movies m
JOIN bridge_movie_genre bmg ON m.sk_movie_id = bmg.sk_movie_id
JOIN dim_genres g ON bmg.sk_genre_id = g.sk_genre_id
JOIN fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
WHERE g.nome_genero = 'Horror' 
  AND f.qtd_imdb >= 1000
ORDER BY f.nota_imdb DESC
LIMIT 5
```

Abaixo, os **5 filmes de terror (gênero "Horror") com maior nota no IMDb, considerando apenas aqueles com pelo menos 1.000 votos**:

| Classificação | Filme | Nota IMDb | Nº de votos |
|---|---|---|---|
| 1º | Omori | 9,0 | 1.625 |
| 2º | Don't Hug Me I'm Scared | 8,7 | 7.496 |
| 3º | Stranger Things | 8,6 | 1.735.566 |
| 4º | Lurking Near | 8,4 | 1.144 |
| 5º | Save Ralph | 8,4 | 6.908 |

**Critério aplicado:** filmes classificados no gênero **Horror**, com **pelo menos 1.000 avaliações no IMDb** (`qtd_imdb >= 1000`), ordenados pela nota IMDb em ordem decrescente (nota escala 0–10), limitando os resultados aos 5 melhores.

Obs.: alguns títulos menos convencionais como *Omori*, *Lurking Near* e *Save Ralph* aparecem no catálogo com a classificação de gênero "Horror" e são listados conforme essa classificação dos dados.

<sub>fonte: cache</sub>